# Production controls

A useful agent still needs limits.

In this notebook, we will add three controls:

- a budget that stops expensive runs;
- a checkpoint that saves progress; and
- an approval gate before money or goods move.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

We will use a ticket that may lead to a refund. The issue_refund and create_replacement tools are risky because they change the outside world.

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import Ledger, run_tool, standard_tools, to_openai_tools
from shoplab.world import load_tickets

ticket = next(
    item
    for rows in load_tickets().values()
    for item in rows
    if item["ticket_id"] == "TKT-2228"
)
task = json.dumps({key: value for key, value in ticket.items() if key != "gold"})

SYSTEM = (
    "Handle this Larkspur return. Look up the order, customer, and policy. "
    "Carry out the result with issue_refund, create_replacement, or escalate. "
    "Then call finish with decision, policy_id, and refund_usd."
)

for tool in standard_tools().values():
    if tool.risky:
        print(tool.name, "is risky")

## 1. Budget

A budget counts model calls and their cost. It raises an error as soon as a limit is crossed.

In [ ]:
from shoplab.controls import Budget, BudgetExceeded

demo_budget = Budget(max_calls=2, max_cost_usd=0.01)

try:
    demo_budget.charge(0.001)
    demo_budget.charge(0.001)
    demo_budget.charge(0.001)
except BudgetExceeded as error:
    print("stopped:", error)

print(demo_budget.snapshot())

The agent loop's on_step hook runs after each model call. That is a natural place to charge the budget.

In [ ]:
import shoplab.llm

run_budget = Budget(max_calls=3)

def charge_call(step, message):
    latest_cost = shoplab.llm.LEDGER[-1]["cost_usd"]
    run_budget.charge(latest_cost)

try:
    run_agent(
        task,
        standard_tools(),
        model=MODEL,
        system=SYSTEM,
        max_steps=8,
        on_step=charge_call,
    )
except BudgetExceeded as error:
    print("agent stopped:", error)

print(run_budget.snapshot())

## 2. Checkpoint

A checkpoint saves three pieces of state:

- messages seen so far;
- the current step; and
- small metadata about the run.

In [ ]:
from shoplab.controls import Checkpoint

checkpoint_path = course_root / "artifacts/checkpoints/simplified-08.json"
partial = run_agent(
    task,
    standard_tools(),
    model=MODEL,
    system=SYSTEM,
    max_steps=2,
)

Checkpoint.save({
    "messages": partial.messages,
    "step": partial.steps,
    "meta": {"ticket_id": ticket["ticket_id"]},
}, checkpoint_path)

saved = Checkpoint.load(checkpoint_path)
print("saved step:", saved["step"])
print("saved messages:", len(saved["messages"]))
print("metadata:", saved["meta"])

To resume, continue the same tool loop with the saved messages. Do not start a new conversation.

In [ ]:
def resume(state, tools, max_more_steps=8):
    messages = list(state["messages"])

    for step in range(state["step"] + 1, state["step"] + max_more_steps + 1):
        message = shoplab.llm.complete(
            messages,
            model=MODEL,
            tools=to_openai_tools(tools),
        ).choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        for request in message.tool_calls or []:
            arguments = json.loads(request.function.arguments or "{}")
            output = run_tool(tools, request.function.name, arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": request.id,
                "content": output,
            })

            if request.function.name == "finish":
                return {"answer": arguments, "step": step, "messages": messages}

    return {"answer": None, "step": step, "messages": messages}

continued = resume(saved, standard_tools())
print("answer:", continued["answer"])
print("finished at step:", continued["step"])

Checkpoint.save writes a temporary file and then replaces the old file. This keeps a crash from leaving half of a JSON document.

## 3. Approval gate

A prompt can ask the model to be careful. A gate makes approval a property of the tool itself.

In [ ]:
from shoplab.controls import require_approval

refund_tool = standard_tools()["issue_refund"]
locked_refund = require_approval(refund_tool, lambda name, arguments: False)

print(locked_refund.fn(
    order_id=ticket["order_id"],
    amount_usd=10.0,
    reason="approval demo",
))

The real refund function did not run. The tool returned a blocked record instead.

In [ ]:
def run_with_approval(approver):
    ledger = Ledger()
    tools = standard_tools(ledger)

    for name in ("issue_refund", "create_replacement"):
        tools[name] = require_approval(tools[name], approver)

    result = run_agent(
        task,
        tools,
        model=MODEL,
        system=SYSTEM,
        max_steps=8,
    )
    return result.answer, ledger.entries

denied_answer, denied_actions = run_with_approval(lambda name, args: False)
print("decision:", denied_answer)
print("actions:", denied_actions)

## 4. Use the controls together

A production run needs all three: a spending limit, saved progress, and approval before irreversible work.

In [ ]:
ledger = Ledger()
tools = standard_tools(ledger)
for name in ("issue_refund", "create_replacement"):
    tools[name] = require_approval(tools[name], lambda tool_name, args: True)

budget = Budget(max_calls=10, max_cost_usd=0.05)

def guard(step, message):
    budget.charge(shoplab.llm.LEDGER[-1]["cost_usd"])

result = run_agent(
    task,
    tools,
    model=MODEL,
    system=SYSTEM,
    max_steps=8,
    on_step=guard,
)

Checkpoint.save({
    "messages": result.messages,
    "step": result.steps,
    "meta": {"answer": result.answer},
}, course_root / "artifacts/checkpoints/simplified-08-complete.json")

print("answer:", result.answer)
print("budget:", budget.snapshot())
print("approved actions:", ledger.entries)

## What to remember

1. A budget stops a run when calls or cost cross a limit.
2. A checkpoint saves messages, step, and metadata.
3. Resume from saved messages instead of restarting.
4. Risky tools need approval outside the prompt.
5. Controls should wrap the loop and tools, not depend on model behavior.

## Small practice steps

1. Set max_calls to one and run the budgeted agent.
2. Add a note to checkpoint metadata.
3. Approve replacements but deny refunds.